In [1]:
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

DATA_DIR = Path.home() / "Downloads" / "transformer_data"
SOURCE_SPLITS = ("Train", "Validation", "Test")
NUM_FRAMES = 37

MODEL_PATH = Path("models/hand_landmarker.task")
SPLIT_FOLDERS = {
    "train": Path("train"),
    "val": Path("validate"),
    "test": Path("test"),
}

_split_lookup = {}
for split_name, csv_path in [
    ("train", "train.csv"),
    ("val", "val.csv"),
    ("test", "test.csv"),
]:
    for video_id in pd.read_csv(csv_path)["video_id"]:
        _split_lookup[video_id] = split_name

_landmarker = None

In [2]:
def find_video(video_id):
    """Return the folder path for a video id under Train, Validation, or Test."""
    for split in SOURCE_SPLITS:
        video_path = DATA_DIR / split / str(video_id)
        if video_path.is_dir():
            return video_path
    raise FileNotFoundError(
        f"Video {video_id} not found in {', '.join(SOURCE_SPLITS)}"
    )


def load_frames(video_path, num_frames=NUM_FRAMES):
    """Load the JPG frames for a video folder."""
    frames = []
    for i in range(1, num_frames + 1):
        frame_path = video_path / f"{i:05d}.jpg"
        frame = cv2.imread(str(frame_path))
        if frame is None:
            raise FileNotFoundError(f"Missing frame: {frame_path}")
        frames.append(frame)
    return frames


def _get_landmarker():
    global _landmarker
    if _landmarker is None:
        MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not MODEL_PATH.exists():
            url = (
                "https://storage.googleapis.com/mediapipe-models/"
                "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
            )
            urllib.request.urlretrieve(url, MODEL_PATH)

        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path=str(MODEL_PATH)),
            running_mode=vision.RunningMode.IMAGE,
            num_hands=1,
            min_hand_detection_confidence=0.3,
        )
        _landmarker = vision.HandLandmarker.create_from_options(options)
    return _landmarker


def extract_landmarks(frames):
    """Run MediaPipe hand landmarks on each frame. Returns shape (37, 63)."""
    landmarker = _get_landmarker()
    landmarks = []

    for frame in frames:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)

        frame_landmarks = np.zeros((21, 3), dtype=np.float32)
        if result.hand_landmarks:
            for i, lm in enumerate(result.hand_landmarks[0]):
                frame_landmarks[i] = [lm.x, lm.y, lm.z]
        landmarks.append(frame_landmarks)

    landmarks = np.stack(landmarks)
    return landmarks.reshape(NUM_FRAMES, 63)


def save_sequence(video_id, landmarks):
    """Save landmarks to train/, validate/, or test/ based on the split CSVs."""
    split = _split_lookup.get(video_id)
    if split is None:
        raise ValueError(f"video_id {video_id} not found in train.csv, val.csv, or test.csv")

    out_dir = SPLIT_FOLDERS[split]
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f"{video_id}.npy", landmarks)

In [3]:
from tqdm import tqdm

for split_folder in SPLIT_FOLDERS.values():
    split_folder.mkdir(parents=True, exist_ok=True)

split_csvs = {
    "train": "train.csv",
    "val": "val.csv",
    "test": "test.csv",
}

video_ids = []
for csv_path in split_csvs.values():
    video_ids.extend(pd.read_csv(csv_path)["video_id"].tolist())

failed = []
for video_id in tqdm(video_ids, desc="Processing videos"):
    split = _split_lookup[video_id]
    out_path = SPLIT_FOLDERS[split] / f"{video_id}.npy"
    if out_path.exists():
        continue

    try:
        video_path = find_video(video_id)
        frames = load_frames(video_path)
        landmarks = extract_landmarks(frames)
        save_sequence(video_id, landmarks)
    except Exception as e:
        failed.append((video_id, str(e)))

print(f"Done. Saved {len(video_ids) - len(failed)} sequences.")
if failed:
    print(f"Failed: {len(failed)}")
    for video_id, error in failed[:10]:
        print(f"  {video_id}: {error}")

Processing videos:   0%|          | 0/15238 [00:00<?, ?it/s]I0000 00:00:1782777488.775421 31594141 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1782777488.859267 31594141 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1782777488.869778 31594143 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782777488.881787 31594143 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782777488.900609 31594145 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
Processing videos:   3%|▎         | 423/15238 [02:00<1:15:

Done. Saved 15238 sequences.


E0000 00:00:1782787031.980306 31594142 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-29T22:44:11.945289-04:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782787091.986155 31594142 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-29T22:44:11.945289-04:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782787151.987520 31594142 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-29T22:44:11.945289-04:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782787211.992427 31594142 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: No